<a href="https://colab.research.google.com/github/dpratap17/DeepLearningLab/blob/main/DL__Exp_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
import numpy as np
import re
import random
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from nltk.translate.bleu_score import corpus_bleu


In [2]:
DATA_PATH = "/content/spa.txt"

def load_data(path, max_samples=10000):
    lines = open(path, encoding="utf-8").read().strip().split("\n")
    random.shuffle(lines)
    lines = lines[:max_samples]

    eng, spa = [], []
    for line in lines:
        e, s = line.split("\t")
        eng.append(e.lower())
        spa.append("<start> " + s.lower() + " <end>")
    return eng, spa

eng_sentences, spa_sentences = load_data(DATA_PATH)


In [3]:
def clean_text(text):
    text = re.sub(r"[^a-zA-Z?.!,¿]+", " ", text)
    return text

eng_sentences = [clean_text(s) for s in eng_sentences]
spa_sentences = [clean_text(s) for s in spa_sentences]

eng_tokenizer = tf.keras.preprocessing.text.Tokenizer(filters="")
spa_tokenizer = tf.keras.preprocessing.text.Tokenizer(filters="")

eng_tokenizer.fit_on_texts(eng_sentences)
spa_tokenizer.fit_on_texts(spa_sentences)

eng_seq = eng_tokenizer.texts_to_sequences(eng_sentences)
spa_seq = spa_tokenizer.texts_to_sequences(spa_sentences)

max_eng = max(len(x) for x in eng_seq)
max_spa = max(len(x) for x in spa_seq)

eng_seq = tf.keras.preprocessing.sequence.pad_sequences(eng_seq, maxlen=max_eng, padding="post")
spa_seq = tf.keras.preprocessing.sequence.pad_sequences(spa_seq, maxlen=max_spa, padding="post")

X_train, X_temp, y_train, y_temp = train_test_split(eng_seq, spa_seq, test_size=0.2)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5)


In [4]:
BATCH_SIZE = 64
BUFFER_SIZE = 20000

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_ds = val_ds.batch(BATCH_SIZE)


In [5]:
class Encoder(tf.keras.Model):
    def __init__(self, vocab, emb, units):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(vocab, emb)
        self.lstm = tf.keras.layers.LSTM(units, return_state=True, return_sequences=True)

    def call(self, x):
        x = self.embedding(x)
        out, h, c = self.lstm(x)
        return h, c


In [6]:
class Decoder(tf.keras.Model):
    def __init__(self, vocab, emb, units):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(vocab, emb)
        self.lstm = tf.keras.layers.LSTM(units, return_sequences=True, return_state=True)
        self.fc = tf.keras.layers.Dense(vocab)

    def call(self, x, h, c):
        x = self.embedding(x)
        out, h, c = self.lstm(x, initial_state=[h, c])
        return self.fc(out), h, c


In [7]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss = loss_fn(real, pred)
    return tf.reduce_mean(loss * tf.cast(mask, tf.float32))


In [8]:
EPOCHS = 10
UNITS = 256
EMB = 256

encoder = Encoder(len(eng_tokenizer.word_index)+1, EMB, UNITS)
decoder = Decoder(len(spa_tokenizer.word_index)+1, EMB, UNITS)

optimizer = tf.keras.optimizers.Adam()

for epoch in range(EPOCHS):
    total_loss = 0
    for x, y in train_ds:
        with tf.GradientTape() as tape:
            h, c = encoder(x)
            dec_input = y[:, :-1]
            real = y[:, 1:]
            preds, _, _ = decoder(dec_input, h, c)
            loss = loss_function(real, preds)

        vars = encoder.trainable_variables + decoder.trainable_variables
        grads = tape.gradient(loss, vars)
        optimizer.apply_gradients(zip(grads, vars))
        total_loss += loss

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_ds):.4f}")


Epoch 1, Loss: 0.3705
Epoch 2, Loss: 0.2120
Epoch 3, Loss: 0.2035
Epoch 4, Loss: 0.1966
Epoch 5, Loss: 0.1910
Epoch 6, Loss: 0.1861
Epoch 7, Loss: 0.1814
Epoch 8, Loss: 0.1767
Epoch 9, Loss: 0.1719
Epoch 10, Loss: 0.1670


In [9]:
class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super().__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)

    def call(self, hidden, enc_out):
        hidden = tf.expand_dims(hidden, 1)
        score = self.V(tf.nn.tanh(self.W1(enc_out) + self.W2(hidden)))
        attn = tf.nn.softmax(score, axis=1)
        context = attn * enc_out
        context = tf.reduce_sum(context, axis=1)
        return context, attn


In [10]:
class LuongAttention(tf.keras.layers.Layer):
    def call(self, hidden, enc_out):
        score = tf.matmul(enc_out, tf.expand_dims(hidden, 2))
        attn = tf.nn.softmax(score, axis=1)
        context = attn * enc_out
        context = tf.reduce_sum(context, axis=1)
        return context, attn


In [11]:
class AttnDecoder(tf.keras.Model):
    def __init__(self, vocab, emb, units, attention):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(vocab, emb)
        self.lstm = tf.keras.layers.LSTM(units, return_sequences=True, return_state=True)
        self.fc = tf.keras.layers.Dense(vocab)
        self.attention = attention

    def call(self, x, h, c, enc_out):
        context, attn = self.attention(h, enc_out)
        x = self.embedding(x)
        context = tf.expand_dims(context, 1)
        x = tf.concat([context, x], axis=-1)
        out, h, c = self.lstm(x, initial_state=[h, c])
        return self.fc(out), h, c, attn


In [13]:
def evaluate_bleu(encoder, decoder):
    references, candidates = [], []

    for x, y in zip(X_test[:200], y_test[:200]):
        h, c = encoder(x[None, :])
        # Corrected: Use "start" instead of "<start>"
        dec_input = tf.constant([[spa_tokenizer.word_index["start"]]])
        result = []

        for _ in range(max_spa):
            preds, h, c = decoder(dec_input, h, c)
            word_id = tf.argmax(preds[0, -1]).numpy()
            # Corrected: Use "end" instead of "<end>"
            if word_id == spa_tokenizer.word_index["end"]:
                break
            result.append(spa_tokenizer.index_word[word_id])
            dec_input = tf.constant([[word_id]])

        references.append([spa_tokenizer.sequences_to_texts([y])[0].split()])
        candidates.append(result)

    return corpus_bleu(references, candidates)

print("BLEU Score:", evaluate_bleu(encoder, decoder))

BLEU Score: 2.1721815141128726e-155


/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


In [16]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

smoothie = SmoothingFunction().method4

references = []
candidates = []

# Use only a small subset for speed
for i in range(200):
    x = X_test[i]
    y = y_test[i]

    # Encoder forward pass
    h, c = encoder(x[None, :])

    # Start token
    dec_input = tf.constant([[spa_tokenizer.word_index["start"]]]) # Corrected from "<start>"
    result = []

    for _ in range(max_spa):
        preds, h, c = decoder(dec_input, h, c)
        word_id = tf.argmax(preds[0, -1]).numpy()

        if word_id == spa_tokenizer.word_index["end"]: # Corrected from "<end>"
            break

        result.append(spa_tokenizer.index_word.get(word_id, ""))
        dec_input = tf.constant([[word_id]])

    # Reference & hypothesis
    ref_sentence = spa_tokenizer.sequences_to_texts([y])[0]
    references.append([ref_sentence.split()])
    candidates.append(result)

# Smoothed BLEU
bleu_score = corpus_bleu(
    references,
    candidates,
    smoothing_function=smoothie
)

print("Smoothed BLEU Score:", bleu_score)

Smoothed BLEU Score: 0.0024824636840713094
